# MAE Pretraining

This notebook is the main walkthrough for the MAE tutorial. You can point it at either the CIFAR-10 or MNIST MAE config from one selector cell below. The Python scripts remain the canonical training implementation, but the conceptual explanation, visual intuition, and experiment guidance live here.


## Environment Notes

- The notebook resolves the repository root automatically.
- Dataset preview cells may download the selected dataset on first use if `download: true` is enabled in the config.
- MNIST is converted to 3-channel RGB inside the datamodule so it can reuse the same MAE pipeline as CIFAR-10.
- Training writes a new timestamped run folder under `outputs/`.

If you need to install the package into the notebook kernel, uncomment the next cell.


In [ ]:
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml")

repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added repo-local src/ to sys.path from {repo_root}.")


## Learning Goals

By the end of this notebook, you should understand:

- what masked autoencoding is trying to learn
- why MAE is a good first tutorial method for either CIFAR-10 or MNIST
- how the YAML config maps onto the actual training behavior
- what the model sees after augmentation and masking
- what artifacts the training run produces for later analysis


## Background And Useful Resources

- MAE paper: <https://arxiv.org/abs/2111.06377>
- MAE reference implementation: <https://github.com/facebookresearch/mae>
- CIFAR-10 dataset page: <https://www.cs.toronto.edu/~kriz/cifar.html>
- MNIST dataset page: <http://yann.lecun.com/exdb/mnist/>
- Lightly MAE example: <https://docs.lightly.ai/self-supervised-learning/examples/mae.html>
- Lightly transforms documentation: <https://docs.lightly.ai/self-supervised-learning/lightly.transforms.html>

A useful mental model for MAE is: **the encoder learns to reason from partial evidence**. Instead of seeing the whole image, it only sees the visible patches and must learn latent structure that helps reconstruct the missing ones.


In [ ]:
import os
from pathlib import Path
from pprint import pprint
import subprocess
import sys

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml")

REPO_ROOT = find_repo_root(Path.cwd())

os.chdir(REPO_ROOT)
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from tutorials.train_ssl import run_training

from fomocid.utils import (
    build_dataset_preview,
    build_mae_mask_preview,
    build_pretrain_view_preview,
    load_config,
    load_config_text,
    summarize_ssl_batch_shapes,
)

AVAILABLE_MAE_CONFIGS = {
    "cifar10": Path("configs/cifar10_mae.yaml"),
    "mnist": Path("configs/mnist_mae.yaml"),
}
SELECTED_DATASET = "mnist" 
if SELECTED_DATASET not in AVAILABLE_MAE_CONFIGS:
    raise ValueError(f"Unsupported dataset selection: {SELECTED_DATASET}")

CONFIG_PATH = AVAILABLE_MAE_CONFIGS[SELECTED_DATASET]
OUTPUT_ROOT = Path("outputs")
print(f"Repo root detected. Running commands relative to the repository root.")
print(f"Selected dataset: {SELECTED_DATASET}")
print(f"Config path: {CONFIG_PATH}")
config = load_config(CONFIG_PATH)
config


## Config Overview

This tutorial is driven by the config selected in `CONFIG_PATH`. Reading the resolved YAML first is a good habit because it keeps the notebook grounded in the actual experiment settings.


In [ ]:
print(load_config_text(CONFIG_PATH))

## What The Selected Dataset Looks Like

Before thinking about transformers, masking, or reconstruction, it helps to anchor ourselves in the data.

- `CIFAR-10` gives small natural images and is useful for learning the full image-benchmark workflow.
- `MNIST` gives simpler handwritten digits and is the fastest real-dataset MAE example in the repository.

Both are useful tutorial datasets, just with different tradeoffs in speed and visual complexity.


In [ ]:
preview = build_dataset_preview(CONFIG_PATH, split="train", limit=10, samples_per_class=1, display_size=224)
preview

## Why MAE Is A Good Fit Here

MAE is especially nice for a tutorial because the task is concrete. We do not need to start with abstract similarity objectives. Instead, we can say:

1. split the image into patches
2. hide many of them
3. encode only the visible subset
4. ask the decoder to reconstruct the missing content

That setup teaches several important ideas at once:

- patch-based vision models
- self-supervision without labels
- the separation between a heavy encoder and a lighter decoder
- why downstream quality must still be measured explicitly

MNIST makes the workflow faster to iterate on, while CIFAR-10 adds more visual diversity once the mechanics feel familiar.


## Masking Intuition

The next cell shows the image, the patch grid, and an example masking pattern. The exact mask changes from sample to sample during training, but the structure of the task stays the same.

In [ ]:
mask_preview = build_mae_mask_preview(CONFIG_PATH, split="train", index=0, seed=config["seed"])
mask_preview

## What The Augmentation Pipeline Produces

Even though MAE is not a multi-view method in the same sense as DINO, it still uses random cropping and flipping. These stochastic views matter because they change which spatial evidence the model sees before masking is even applied.

In [ ]:
view_preview = build_pretrain_view_preview(CONFIG_PATH, split="train", index=0, repeats=4, display_size=224)
view_preview

## What The DataLoader Actually Returns

One of the easiest ways to demystify a training script is to inspect the batch structure directly. For MAE, we expect a single augmented image tensor per sample rather than a list of many crops.

In [ ]:
pprint(summarize_ssl_batch_shapes(CONFIG_PATH))

## Important Parameters And Why They Matter

### `dataset`

- `image_size`: controls the spatial resolution the model sees
- `batch_size`: affects optimization stability and throughput
- `eval_batch_size`: controls how fast later probing and analysis run

### `ssl`

- `patch_size`: smaller patches create more tokens and a finer reconstruction problem
- `mask_ratio`: higher masking makes the pretext task harder and more information-starved
- `encoder_depth`, `encoder_num_heads`, `encoder_hidden_dim`, `encoder_mlp_dim`: control encoder capacity
- `decoder_depth`, `decoder_num_heads`, `decoder_hidden_dim`, `decoder_mlp_dim`: control reconstruction capacity
- `normalize_pixel_targets`: standard MAE-style target normalization for more stable reconstruction loss

### `optimizer` and `trainer`

- `lr`, `weight_decay`: AdamW optimization settings
- `max_epochs`: how long pretraining runs
- `limit_train_batches`: useful for debugging without a full run
- `accelerator`, `devices`: Lightning hardware selection

In [ ]:
image_size = config["dataset"]["image_size"]
patch_size = config["ssl"]["patch_size"]
mask_ratio = config["ssl"]["mask_ratio"]
patches_per_side = image_size // patch_size
num_patches = patches_per_side ** 2
num_visible = int(num_patches * (1.0 - mask_ratio))
print(f"image_size={image_size}")
print(f"patch_size={patch_size}")
print(f"patches_per_side={patches_per_side}")
print(f"total_patches={num_patches}")
print(f"approx_visible_patches={num_visible}")
print(f"mask_ratio={mask_ratio}")

## Script Interface

The notebook is meant to teach the workflow, but the script is still the source of truth. It is worth reading the CLI help because it shows the official inputs and outputs of the training entrypoint.

In [ ]:
_ = subprocess.run([sys.executable, "tutorials/train_ssl.py", "--help"], check=False, cwd=str(REPO_ROOT))


## What Happens During Training

The canonical command is:

```bash
python tutorials/train_ssl.py --config configs/<selected-mae-config>.yaml --output-dir outputs
```

For the current notebook selection, the script will:

1. load the selected MAE config
2. seed the run
3. build the matching datamodule and MAE module
4. train with Lightning
5. write a timestamped run folder containing logs, checkpoints, and a run summary

The next cell calls `run_training(...)` directly in the notebook kernel so the progress bar can update in place instead of printing a new line for every refresh.


In [ ]:
run_dir = run_training(CONFIG_PATH, OUTPUT_ROOT)
print(run_dir)
print("Training finished. Artifacts were written under outputs/.")


In [ ]:
run_prefix = f"{config['dataset']['name']}_{config['ssl']['method']}_"
checkpoints = sorted(OUTPUT_ROOT.glob(f"{run_prefix}*/checkpoints/last.ckpt"), key=lambda path: path.stat().st_mtime)
latest_checkpoint = checkpoints[-1] if checkpoints else None
latest_checkpoint


## What To Try Next

After one successful run, useful follow-up experiments are:

- switch between `mnist` and `cifar10` to compare speed versus visual complexity
- reduce `mask_ratio` if reconstruction is too hard early on
- increase `encoder_depth` once the workflow is stable
- increase `max_epochs` before changing many architecture settings at once
- compare later representations in the analysis notebook rather than relying on training loss alone

Continue with the `02_mae_analysis.ipynb` notebook to inspect the resulting embeddings.
